# NIG-TimeNet (PyTorch): train and test

**Runtime > Run all.** `CHANGES` is a dict of dotted keys on `RunSettings` (`nig_timenet_torch/workflow/settings.py`):
`{"kind": "flat" | "panel", "train.epochs": 40, "targets.mode": "scaled", ...}`. A misspelt key fails with a suggestion.
Set `ROLLING` to run the same model fold after fold (`train_span`, `val_span`, `test_span`, `step`, `embargo`, `init`
`"fresh"` or `"warm"`, `seeds`). `RETRAIN` re-trains the finished run with a change, for example another target
normalisation: `{"targets.mode": "scaled"}`.

The data is read by name: `DATA_NAME = "preprocessing_output"` adds `_latest` (`preprocessing_output_20261027` reads
`preprocessing_output_20261027_latest.pkl.gz`), and a full name (`preprocessing_output_20261027_153000.pkl.gz` or
`....dataset`) is read as written.

In [ ]:
TORCH_REPO, TORCH_REF = "yuosef772424/nig-timenet-torch", "feat/native-data-notebooks-uncertainty"
EXPECT_DEVICE = "cuda"                         # "cpu" only for a deliberate CPU run; a cuda run on a CPU VM fails
TOKEN_PATH = "/content/drive/MyDrive/crypto/secrets/github_token.txt"
WORKDIR = "/content/nig_runner"                # clones' scratch, logs, git credentials helper: never /tmp (rule E2)
TORCH_DIR = "/content/nig-timenet-torch"
DRIVE_RUNS = "/content/drive/MyDrive/nig_timenet_runs"
EXTRA_MODULES = (("pyarrow", "pyarrow"),)
DATA_DIR = "/content/drive/MyDrive/crypto_model/preprocessed_data"
DATA_NAME = "preprocessing_output"             # base name (+ _latest) or the full file / folder name
RUN_DIR = "/content/drive/MyDrive/nig_timenet_runs/main"
CHANGES = {"kind": "flat", "train.epochs": 30, "train.batch_size": 256}
ROLLING = {}                                   # e.g. {"train_span": "365D", "val_span": "30D", "test_span": "30D", "step": "30D", "embargo": "4h", "init": "warm"}
RETRAIN = {}                                   # e.g. {"targets.mode": "scaled"}; empty = no second run

## 1. Setup
The first cell is plain code: nothing of the package exists on the VM yet, so it downloads the one stdlib-only file
`colab/bootstrap.py` with the token and runs it. Everything after it is a normal import.

In [ ]:
import getpass, os, urllib.error, urllib.request
from google.colab import drive
drive.mount("/content/drive")
token = open(TOKEN_PATH).read().strip() if os.path.exists(TOKEN_PATH) else getpass.getpass("GitHub token (not shown): ")
url = f"https://api.github.com/repos/{TORCH_REPO}/contents/src/nig_timenet_torch/colab/bootstrap.py?ref={TORCH_REF}"
req = urllib.request.Request(url, headers={"Authorization": "Bearer " + token.strip(), "User-Agent": "nig-colab",
                                           "Accept": "application/vnd.github.raw"})
try:
    boot_source = urllib.request.urlopen(req, timeout=60).read().decode()
except urllib.error.HTTPError as e:
    raise SystemExit(f"GitHub answered HTTP {e.code}: the token is missing, expired or cannot read {TORCH_REPO}. "
                     f"Delete {TOKEN_PATH} and run this cell again.")
boot = {}
exec(compile(boot_source, "bootstrap.py", "exec"), boot)
token = boot["acquire_token"](TOKEN_PATH, TORCH_REPO, lambda: getpass.getpass("GitHub token (not shown): "))

In [ ]:
shas = boot["setup_clones"](token, WORKDIR, {"torch": (TORCH_REPO, TORCH_REF, TORCH_DIR)})
install = boot["install_package"](TORCH_DIR, modules=EXTRA_MODULES)

In [ ]:
from nig_timenet_torch.colab.session import check_environment
setup = check_environment(EXPECT_DEVICE, shas, install, WORKDIR, DRIVE_RUNS)

## 2. Data and settings

In [ ]:
from nig_timenet_torch.core.config import updated
from nig_timenet_torch.data_io.loading import load_dataset
from nig_timenet_torch.workflow.settings import RunSettings
paths = {"data.directory": DATA_DIR, "data.name": DATA_NAME, "run_dir": RUN_DIR}
settings = updated(RunSettings(), {**paths, **CHANGES, **({"rolling": ROLLING} if ROLLING else {})})
loaded = load_dataset(settings.data)
print(loaded.path, loaded.format, len(loaded.dataset["last_candles"]), "windows")

## 3. Train and test
One call. A single split trains on the train part, selects on validation and scores the test part; with `ROLLING` every
fold is trained, selected and scored in time order (`init="warm"` continues the previous fold's model).

In [ ]:
from nig_timenet_torch.workflow.run import run
result = run(settings, loaded)

## 4. Results

In [ ]:
from nig_timenet_torch.workflow.run import rolling_summary, scalars
print(rolling_summary(result) if settings.rolling else scalars(result.report))

## 5. Re-train with a change
Runs again from the same data with `RETRAIN` applied to the settings (a new target normalisation, learning rate, ...).

In [ ]:
from nig_timenet_torch.workflow.run import retrain
second = retrain(result, RETRAIN) if RETRAIN else None
print(scalars(second.report) if second and second.report else "no re-train requested")